# Western Ghats Research — Step 5: Train MLP Student Head

This notebook trains a **Residual MLP Head** on the full competition vocabulary (182 species).

Key Upgrades Included:
- **Logit-Adjusted Focal Loss**
- **Focal-Soundscape Mixup**
- **Per-Class Temperature Calibration**
- **Full 182-Species Output** (Handles distractor birds)

## 0. Environment & Debug Flag

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

# -----------------------------------------------------------------------------
# DEBUG_MODE: set True when running locally.
# On Kaggle: set False -> runs actual training.
# -----------------------------------------------------------------------------

INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = Path('.')

C:\Users\piyus\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'sklearn'

## 1. Model Components

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x): return x + self.net(x)

class MLPHead(nn.Module):
    def __init__(self, in_dim=1536, hidden_dim=512, n_classes=182, dropout=0.3):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.res_block = ResidualBlock(hidden_dim, dropout)
        self.output_layer = nn.Linear(hidden_dim, n_classes)
        self.temperature = nn.Parameter(torch.ones(n_classes))
    def forward(self, x, apply_calibration=True):
        x = self.input_layer(x)
        x = self.res_block(x)
        logits = self.output_layer(x)
        if apply_calibration:
            t = torch.nn.functional.softplus(self.temperature) + 1e-4
            logits = logits / t
        return logits

class LogitAdjustedFocalBCE(nn.Module):
    def __init__(self, log_prior, tau=1.0, gamma=2.0, alpha=0.25):
        super().__init__()
        self.register_buffer("log_prior", log_prior)
        self.tau = tau; self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        logits_adj = logits + self.tau * self.log_prior
        bce_loss = F.binary_cross_entropy_with_logits(logits_adj, targets, reduction='none')
        p = torch.sigmoid(logits_adj)
        p_t = p * targets + (1 - p) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        alpha_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_weight * focal_weight * bce_loss).mean()

class MixupEmbeddingDataset(Dataset):
    def __init__(self, focal_emb, focal_y, sound_emb=None, sound_y=None, alpha=0.4, p_mix=0.5):
        self.focal_emb = torch.tensor(focal_emb, dtype=torch.float32)
        self.focal_y = torch.tensor(focal_y, dtype=torch.float32)
        self.sound_emb = torch.tensor(sound_emb, dtype=torch.float32) if sound_emb is not None else None
        self.sound_y = torch.tensor(sound_y, dtype=torch.float32) if sound_y is not None else None
        self.alpha = alpha; self.p_mix = p_mix
    def __len__(self): return len(self.focal_emb)
    def __getitem__(self, idx):
        x, y = self.focal_emb[idx], self.focal_y[idx]
        if self.sound_emb is not None and np.random.rand() < self.p_mix:
            j = np.random.randint(0, len(self.sound_emb))
            lam = np.random.beta(self.alpha, self.alpha)
            x = lam * x + (1 - lam) * self.sound_emb[j]
            y = lam * y + (1 - lam) * self.sound_y[j]
        return x, y

## 2. Path & Configuration

In [ ]:
TRAIN_METADATA = INPUT_DIR / 'bird-data' / 'train_metadata.csv'

EPOCHS = 50
BATCH_SIZE = 256
LR = 1e-3
TAU = 1.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

meta_df = pd.read_csv(TRAIN_METADATA)
ALL_SPECIES = sorted(meta_df['primary_label'].unique())
N_CLASSES = len(ALL_SPECIES)
print(f"Competition Vocabulary: {N_CLASSES} species.")

## 3. Data Loading

In [ ]:
# 3.1 Load Focal Embeddings (All 182 Species)
train_files = list(INPUT_DIR.rglob('perch_train*.parquet'))
if train_files:
    train_df = pd.concat([pd.read_parquet(f) for f in train_files]).reset_index(drop=True)
    # Filter to only include species in the official 182 list
    train_df = train_df[train_df['species_code'].isin(ALL_SPECIES)]
    
    focal_emb = np.stack(train_df['emb'].values)
    focal_y = np.zeros((len(focal_emb), N_CLASSES))
    for i, sp in enumerate(train_df['species_code']):
        focal_y[i, ALL_SPECIES.index(sp)] = 1
    print(f"Loaded {len(focal_emb)} focal embeddings for all species.")
else:
    print("No training embeddings found! Check dataset attachments.")

# 3.2 Load Pseudo-Labels (Western Ghats Soundscapes)
pseudo_files = list(INPUT_DIR.rglob('round1_pseudo_labels.parquet'))
if pseudo_files:
    pseudo_df = pd.read_parquet(pseudo_files[0])
    sound_emb = np.stack(pseudo_df['emb'].values)
    sound_y = np.stack(pseudo_df['pseudo_label'].values)
    print(f"Loaded {len(sound_emb)} pseudo-labeled chunks.")
else:
    print("No pseudo-labels found. Using focal data only.")
    sound_emb, sound_y = None, None

## 4. Training

In [ ]:
class_counts = focal_y.sum(axis=0) + (sound_y.sum(axis=0) if sound_y is not None else 1)
log_prior = torch.log(torch.tensor(class_counts / class_counts.sum()) + 1e-12).to(DEVICE)

dataset = MixupEmbeddingDataset(focal_emb, focal_y, sound_emb, sound_y)
freq = focal_y.sum(axis=0) + 1
sample_weights = (focal_y * (1.0 / freq)).sum(axis=1)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)

model = MLPHead(in_dim=focal_emb.shape[1], n_classes=N_CLASSES).to(DEVICE)
criterion = LogitAdjustedFocalBCE(log_prior, tau=TAU)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x, apply_calibration=False), y)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {epoch_loss/len(dataloader):.4f}")

## 5. Calibration & Savings

In [ ]:
# Quick Calibration on focal train subset
model.eval()
with torch.no_grad():
    v_logits = model(torch.tensor(focal_emb[:2000]).to(DEVICE), apply_calibration=False)
    v_targets = torch.tensor(focal_y[:2000]).to(DEVICE)

t_opt = torch.optim.Adam([model.temperature], lr=0.01)
for _ in range(100):
    t_opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(v_logits / (F.softplus(model.temperature)+1e-4), v_targets)
    loss.backward(); t_opt.step()

torch.save(model.state_dict(), 'mlp_head_182.pth')
print("Training Complete. Model saved as mlp_head_182.pth")